25.12.3
新增保存实验结果
保存了完整的实验结果

In [29]:
import torch
import torch.nn as nn
import os
from transformers import AutoModel, AutoTokenizer
from transformers import BertModel, BertTokenizer
import random
from scipy.interpolate import interp1d
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score,roc_curve
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import random
import matplotlib.pyplot as plt
from torch.utils.data.sampler import SubsetRandomSampler
import numpy as np
import torch
import csv
import pandas as pd
import pickle
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, matthews_corrcoef, precision_score, recall_score, f1_score
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, 
                             matthews_corrcoef, precision_score, recall_score, 
                             f1_score, confusion_matrix)


In [2]:
#Model settings
batch_size = 32  # Setting batchsize
learning_rates=0.000065 # Setting learning rates
model_name="esm2_6"#Select different versions of the ESM2 model: esm2_6,esm_12,esm_30.

In [3]:
#Dataset definitions

class MyDataset(Dataset):
    def __init__(self, file):
        self.sequence, self.label = self.read_file(file)
        self.sequence_protbert=self.add_space_between_characters(self.sequence)

    def read_file(self,file_path):
        sequences = []
        labels = []
        with open(file_path, 'r', newline='') as csv_file:
            csv_reader = csv.reader(csv_file)
            next(csv_reader, None)  
            data = list(csv_reader)
            random.seed(42)
            random.shuffle(data)  
            for row in data:
                sequences.append(row[1])
                labels.append(row[2])
        return sequences, labels
    
    def add_space_between_characters(self,input_list):
        new_list = []
        for element in input_list:
            new_element = ' '.join(element)
            new_list.append(new_element)
        return new_list

    def __len__(self):
        return len(self.sequence)

    def __getitem__(self, index):
        sample=self.sequence[index]
        sample_protbert=self.sequence_protbert[index]
        label=int(self.label[index])
        return sample, label, sample_protbert

In [4]:
# Read the training set

train_file = 'E:\\LLM+XWT\\XWT数据\\Umami-BERT-UMP789\\1_1-data.csv'  
train_dataset = MyDataset(train_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

In [5]:
#Define the esm family of models

class MyModel(nn.Module):
    def __init__(self,):
        super(MyModel, self).__init__()
        if model_name=="esm2_30":
            self.model = AutoModel.from_pretrained("facebook/esm2_t30_150M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D")
            self.layer=640
        elif model_name=="esm2_12":
            self.model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
            self.layer=480
        elif model_name=="esm2_6":
            self.model = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
            self.layer=320
        self.fc2 = nn.Linear(self.layer, 2) 

    def forward(self, inputs,inputs2):
        inputs = self.tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output1 = outputs.pooler_output   
        x=self.fc2(pooler_output1)
        return x

In [30]:
#Model loading and setting

device = torch.device("cpu") 
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model = MyModel()
model.to(device)#Model loading
criterion = nn.CrossEntropyLoss()
loss_all=99999
best_auc=0
all_fpr = []
all_tpr = []
all_aucs = []

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# --- 2. 训练主循环 (核心修改部分) ---

# 数据加载 (请修改为你实际的路径)
train_file = 'E:\\LLM+XWT\\XWT数据\\Umami-BERT-UMP789\\1_1-data.csv' 
dataset = MyDataset(train_file) # 使用整个数据集，后面KFold会切分

from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=False)

# 用于存储所有Fold的所有Epoch数据
all_folds_history = {} 
# 用于存储原本的ROC数据 (最佳Epoch)
roc_data = {'all_fpr': [], 'all_tpr': [], 'all_aucs': []}

epochs = 50 # 谢等人用了10个Epoch，你可以根据收敛情况调整，建议10-20即可

print(f"Start training model: {model_name}")

for fold, (train_indices, valid_indices) in enumerate(kf.split(dataset)):
    print(f"------ Fold {fold + 1} ------")
    
    # 初始化记录器
    history = {
        'train_loss': [], 'val_loss': [], 
        'val_auc': [], 'val_acc': [],
        'val_mcc': [], 'val_f1': [],
        'val_sn': [],  'val_sp': [], 'val_prec': []
    }
    
    train_sampler = SubsetRandomSampler(train_indices)
    valid_sampler = SubsetRandomSampler(valid_indices)
    train_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
    valid_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=valid_sampler)
    
    model = MyModel()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rates)
    criterion = nn.CrossEntropyLoss()
    
    best_auc_fold = 0
    best_fpr_fold = None
    best_tpr_fold = None
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss_accum = 0
        for batch_data, batch_labels, batch_prot in train_dataloader:
            batch_labels = batch_labels.to(device)
            # 注意：如果模型不需要protbert输入，只传batch_data
            outputs = model(batch_data, batch_prot) 
            loss = criterion(outputs, batch_labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss_accum += loss.item()
            
        avg_train_loss = train_loss_accum / len(train_dataloader)
        
        # --- Validation Phase ---
        model.eval()
        val_loss_accum = 0
        all_labels = []
        all_probs = [] # 存储正类的概率
        all_preds = [] # 存储预测类别(0或1)
        
        with torch.no_grad():
            for batch_data, batch_labels, batch_prot in valid_dataloader:
                batch_labels = batch_labels.to(device)
                outputs = model(batch_data, batch_prot)
                loss = criterion(outputs, batch_labels)
                val_loss_accum += loss.item()
                
                probs = torch.softmax(outputs, dim=1)[:, 1] # 取label=1的概率
                preds = torch.argmax(outputs, dim=1)
                
                all_labels.extend(batch_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
        
        avg_val_loss = val_loss_accum / len(valid_dataloader)
        
        # 2. 计算所有指标 (ALL Metrics Calculation)
        # ------------------------------------------------------
        val_auc = roc_auc_score(all_labels, all_probs)
        val_acc = accuracy_score(all_labels, all_preds)
        val_mcc = matthews_corrcoef(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds)
        val_prec = precision_score(all_labels, all_preds, zero_division=0)
        val_sn = recall_score(all_labels, all_preds, zero_division=0) # Recall 就是 Sensitivity
        
        # 计算 Specificity (SP)
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
        val_sp = tn / (tn + fp) if (tn + fp) > 0 else 0
        # ------------------------------------------------------

        # 3. 保存所有指标到 history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
        history['val_acc'].append(val_acc)
        history['val_mcc'].append(val_mcc)
        history['val_f1'].append(val_f1)
        history['val_prec'].append(val_prec)
        history['val_sn'].append(val_sn)
        history['val_sp'].append(val_sp)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_train_loss:.4f} | AUC: {val_auc:.4f} | ACC: {val_acc:.4f} | MCC: {val_mcc:.4f}")
        

        # 保存最佳ROC数据
        if val_auc > best_auc_fold:
            best_auc_fold = val_auc
            fpr, tpr, _ = roc_curve(all_labels, all_probs)
            best_fpr_fold = fpr
            best_tpr_fold = tpr
            # 你也可以在这里计算并保存最佳Epoch的所有其他指标(MCC, SN, SP等)用于画Fig 3E
            
    # Fold结束，保存该Fold的ROC信息
    roc_data['all_fpr'].append(best_fpr_fold)
    roc_data['all_tpr'].append(best_tpr_fold)
    roc_data['all_aucs'].append(best_auc_fold)
    
    # 保存该Fold的历史记录
    all_folds_history[f'fold_{fold+1}'] = history

# --- 3. 保存所有数据 ---
save_dir = 'E:/LLM+XWT/最终版代码/训练结果/训练过程2_6/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 保存完整训练记录（用于画谢等人的 Fig 3 A-D）
with open(os.path.join(save_dir, f'{model_name}_history.pkl'), 'wb') as f:
    pickle.dump(all_folds_history, f)

# 保存ROC数据（用于画你原来的 Fig 4）
with open(os.path.join(save_dir, f'{model_name}_roc_data.pkl'), 'wb') as f:
    pickle.dump(roc_data, f)
    
print("All data saved.")

Start training model: esm2_6
------ Fold 1 ------


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5896 | AUC: 0.9294 | ACC: 0.8293 | MCC: 0.6845
Epoch 2/50 | Loss: 0.3353 | AUC: 0.9500 | ACC: 0.8780 | MCC: 0.7569
Epoch 3/50 | Loss: 0.2850 | AUC: 0.9558 | ACC: 0.8780 | MCC: 0.7571
Epoch 4/50 | Loss: 0.2867 | AUC: 0.9500 | ACC: 0.8943 | MCC: 0.7936
Epoch 5/50 | Loss: 0.2319 | AUC: 0.9471 | ACC: 0.8537 | MCC: 0.7077
Epoch 6/50 | Loss: 0.1404 | AUC: 0.9714 | ACC: 0.8618 | MCC: 0.7237
Epoch 7/50 | Loss: 0.1271 | AUC: 0.9640 | ACC: 0.8780 | MCC: 0.7588
Epoch 8/50 | Loss: 0.0895 | AUC: 0.9640 | ACC: 0.8780 | MCC: 0.7562
Epoch 9/50 | Loss: 0.0439 | AUC: 0.9625 | ACC: 0.8943 | MCC: 0.7887
Epoch 10/50 | Loss: 0.0365 | AUC: 0.9569 | ACC: 0.8943 | MCC: 0.7911
Epoch 11/50 | Loss: 0.0223 | AUC: 0.9617 | ACC: 0.8862 | MCC: 0.7723
Epoch 12/50 | Loss: 0.0139 | AUC: 0.9688 | ACC: 0.8943 | MCC: 0.7887
Epoch 13/50 | Loss: 0.0072 | AUC: 0.9635 | ACC: 0.9106 | MCC: 0.8212
Epoch 14/50 | Loss: 0.0046 | AUC: 0.9651 | ACC: 0.9106 | MCC: 0.8212
Epoch 15/50 | Loss: 0.0033 | AUC: 0.9662 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5969 | AUC: 0.9272 | ACC: 0.8455 | MCC: 0.6908
Epoch 2/50 | Loss: 0.3981 | AUC: 0.9473 | ACC: 0.8618 | MCC: 0.7551
Epoch 3/50 | Loss: 0.2695 | AUC: 0.9454 | ACC: 0.8862 | MCC: 0.7742
Epoch 4/50 | Loss: 0.2085 | AUC: 0.9653 | ACC: 0.8943 | MCC: 0.7916
Epoch 5/50 | Loss: 0.1647 | AUC: 0.9703 | ACC: 0.9024 | MCC: 0.8047
Epoch 6/50 | Loss: 0.1135 | AUC: 0.9674 | ACC: 0.8780 | MCC: 0.7626
Epoch 7/50 | Loss: 0.1249 | AUC: 0.9690 | ACC: 0.8780 | MCC: 0.7574
Epoch 8/50 | Loss: 0.1136 | AUC: 0.9650 | ACC: 0.8780 | MCC: 0.7596
Epoch 9/50 | Loss: 0.0607 | AUC: 0.9642 | ACC: 0.9268 | MCC: 0.8567
Epoch 10/50 | Loss: 0.0379 | AUC: 0.9611 | ACC: 0.9106 | MCC: 0.8208
Epoch 11/50 | Loss: 0.0318 | AUC: 0.9653 | ACC: 0.9268 | MCC: 0.8567
Epoch 12/50 | Loss: 0.0195 | AUC: 0.9709 | ACC: 0.9187 | MCC: 0.8374
Epoch 13/50 | Loss: 0.0359 | AUC: 0.9656 | ACC: 0.8943 | MCC: 0.7882
Epoch 14/50 | Loss: 0.0446 | AUC: 0.9682 | ACC: 0.9106 | MCC: 0.8208
Epoch 15/50 | Loss: 0.0623 | AUC: 0.9772 | 

...

Epoch 1/50 | Loss: 0.5331 | AUC: 0.9371 | ACC: 0.8862 | MCC: 0.7779
Epoch 2/50 | Loss: 0.3468 | AUC: 0.9323 | ACC: 0.8943 | MCC: 0.7880
Epoch 3/50 | Loss: 0.2550 | AUC: 0.9515 | ACC: 0.8943 | MCC: 0.7969
Epoch 4/50 | Loss: 0.1823 | AUC: 0.9510 | ACC: 0.9187 | MCC: 0.8388
Epoch 5/50 | Loss: 0.0908 | AUC: 0.9603 | ACC: 0.8780 | MCC: 0.7630
Epoch 6/50 | Loss: 0.0525 | AUC: 0.9571 | ACC: 0.9024 | MCC: 0.8055
Epoch 7/50 | Loss: 0.0325 | AUC: 0.9494 | ACC: 0.8537 | MCC: 0.7109
Epoch 8/50 | Loss: 0.1191 | AUC: 0.9416 | ACC: 0.8943 | MCC: 0.7867
Epoch 9/50 | Loss: 0.1282 | AUC: 0.9531 | ACC: 0.8862 | MCC: 0.7735
Epoch 10/50 | Loss: 0.0435 | AUC: 0.9552 | ACC: 0.8780 | MCC: 0.7549
Epoch 11/50 | Loss: 0.0285 | AUC: 0.9443 | ACC: 0.9024 | MCC: 0.8062
Epoch 12/50 | Loss: 0.0059 | AUC: 0.9483 | ACC: 0.9187 | MCC: 0.8361
Epoch 13/50 | Loss: 0.0042 | AUC: 0.9486 | ACC: 0.9024 | MCC: 0.8039
Epoch 14/50 | Loss: 0.0033 | AUC: 0.9486 | ACC: 0.9024 | MCC: 0.8039
Epoch 15/50 | Loss: 0.0026 | AUC: 0.9491 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5822 | AUC: 0.9730 | ACC: 0.9187 | MCC: 0.8391
Epoch 2/50 | Loss: 0.3562 | AUC: 0.9672 | ACC: 0.8862 | MCC: 0.7928
Epoch 3/50 | Loss: 0.3155 | AUC: 0.9744 | ACC: 0.9350 | MCC: 0.8703
Epoch 4/50 | Loss: 0.1847 | AUC: 0.9699 | ACC: 0.9431 | MCC: 0.8863
Epoch 5/50 | Loss: 0.1639 | AUC: 0.9733 | ACC: 0.9431 | MCC: 0.8872
Epoch 6/50 | Loss: 0.1344 | AUC: 0.9685 | ACC: 0.9268 | MCC: 0.8546
Epoch 7/50 | Loss: 0.1009 | AUC: 0.9730 | ACC: 0.9512 | MCC: 0.9043
Epoch 8/50 | Loss: 0.0641 | AUC: 0.9691 | ACC: 0.9512 | MCC: 0.9029
Epoch 9/50 | Loss: 0.0430 | AUC: 0.9688 | ACC: 0.9268 | MCC: 0.8537
Epoch 10/50 | Loss: 0.0329 | AUC: 0.9638 | ACC: 0.8862 | MCC: 0.7762
Epoch 11/50 | Loss: 0.0170 | AUC: 0.9688 | ACC: 0.9024 | MCC: 0.8116
Epoch 12/50 | Loss: 0.0086 | AUC: 0.9707 | ACC: 0.9268 | MCC: 0.8538
Epoch 13/50 | Loss: 0.0053 | AUC: 0.9667 | ACC: 0.9187 | MCC: 0.8379
Epoch 14/50 | Loss: 0.0035 | AUC: 0.9696 | ACC: 0.9187 | MCC: 0.8379
Epoch 15/50 | Loss: 0.0023 | AUC: 0.9691 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5849 | AUC: 0.8596 | ACC: 0.7213 | MCC: 0.5466
Epoch 2/50 | Loss: 0.3602 | AUC: 0.8980 | ACC: 0.7623 | MCC: 0.5409
Epoch 3/50 | Loss: 0.2640 | AUC: 0.9336 | ACC: 0.8525 | MCC: 0.7064
Epoch 4/50 | Loss: 0.2075 | AUC: 0.9296 | ACC: 0.8115 | MCC: 0.6785
Epoch 5/50 | Loss: 0.1363 | AUC: 0.9425 | ACC: 0.8607 | MCC: 0.7199
Epoch 6/50 | Loss: 0.0712 | AUC: 0.9366 | ACC: 0.8607 | MCC: 0.7270
Epoch 7/50 | Loss: 0.0714 | AUC: 0.9366 | ACC: 0.8443 | MCC: 0.6869
Epoch 8/50 | Loss: 0.0592 | AUC: 0.9447 | ACC: 0.8607 | MCC: 0.7255
Epoch 9/50 | Loss: 0.1022 | AUC: 0.9530 | ACC: 0.8770 | MCC: 0.7547
Epoch 10/50 | Loss: 0.0756 | AUC: 0.9422 | ACC: 0.8607 | MCC: 0.7211
Epoch 11/50 | Loss: 0.0219 | AUC: 0.9433 | ACC: 0.8689 | MCC: 0.7375
Epoch 12/50 | Loss: 0.0211 | AUC: 0.9484 | ACC: 0.8525 | MCC: 0.7047
Epoch 13/50 | Loss: 0.0146 | AUC: 0.9439 | ACC: 0.8361 | MCC: 0.6738
Epoch 14/50 | Loss: 0.0302 | AUC: 0.9398 | ACC: 0.8607 | MCC: 0.7201
Epoch 15/50 | Loss: 0.0207 | AUC: 0.9363 | 